In [1]:
# Imbalance Extension: I1 Linear Winner + Random Oversampling
# Based on winning linear family model under final horse-race plan v3
# Base model = Elastic-Net Logistic (winsorized), C=0.1, l1_ratio=0.5
# This script evaluates the fixed winner under a new imbalance treatment only.

import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*'penalty' was deprecated.*",
    category=FutureWarning,
)

import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

CV_FOLD_METRICS_CSV = BASE_DIR / "Model_I1_LinearWinner_RandomOversampling_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_I1_LinearWinner_RandomOversampling_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_I1_LinearWinner_RandomOversampling_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Model_I1_LinearWinner_RandomOversampling_v1_Coefficients.csv"

# Locked winning linear family hyperparameters
SELECTED_C = 0.1
SELECTED_L1_RATIO = 0.5
MODEL_RANDOM_STATE = 42
OVERSAMPLING_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = "targeted_in_year"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LINEAR v4)
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR BUILDER
# =========================================================
def build_preprocessor():
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

# =========================================================
# 8. MODEL BUILDER
# =========================================================
def build_model():
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=SELECTED_C,
        l1_ratio=SELECTED_L1_RATIO,
        max_iter=20000,
        random_state=MODEL_RANDOM_STATE,
    )

# =========================================================
# 9. RANDOM OVERSAMPLING HELPER
# =========================================================
def random_oversample_to_balance(X, y, random_state=42):
    X = np.asarray(X)
    y = np.asarray(y).astype(int)

    class_0_idx = np.where(y == 0)[0]
    class_1_idx = np.where(y == 1)[0]

    if len(class_0_idx) == 0 or len(class_1_idx) == 0:
        return X, y

    majority_idx = class_0_idx if len(class_0_idx) >= len(class_1_idx) else class_1_idx
    minority_idx = class_1_idx if len(class_0_idx) >= len(class_1_idx) else class_0_idx

    n_majority = len(majority_idx)
    n_minority = len(minority_idx)

    if n_majority == n_minority:
        return X, y

    rng = np.random.default_rng(random_state)
    sampled_minority_idx = rng.choice(minority_idx, size=n_majority, replace=True)

    balanced_idx = np.concatenate([majority_idx, sampled_minority_idx])
    rng.shuffle(balanced_idx)

    return X[balanced_idx], y[balanced_idx]

# =========================================================
# 10. FEATURE NAME CLEANUP
# =========================================================
def clean_feature_names(feature_names):
    cleaned = []
    for name in feature_names:
        cleaned_name = name
        for prefix in ["cont__", "bin__", "ff17__"]:
            if cleaned_name.startswith(prefix):
                cleaned_name = cleaned_name[len(prefix):]
        cleaned.append(cleaned_name)
    return cleaned

# =========================================================
# 11. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 12. EVALUATE FIXED SELECTED MODEL UNDER NEW IMBALANCE TREATMENT
# =========================================================
start_time = time.time()
selected_cv_rows = []
prediction_frames = []

print("I1 Linear Winner + Random Oversampling: starting six-fold evaluation...")

for fold_num, fold_def in enumerate(cv_fold_defs, start=1):
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train_raw = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy().astype(int)

    X_val_raw = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy().astype(int)

    print(
        f"[CV] Fold {fold_num}/6 | train={min(train_years)}-{max(train_years)} "
        f"| val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
    )

    preprocessor = build_preprocessor()
    X_train = preprocessor.fit_transform(X_train_raw, y_train)
    X_val = preprocessor.transform(X_val_raw)

    orig_train_rows = X_train.shape[0]
    orig_train_positives = int(y_train.sum())


    X_train_final, y_train_final = random_oversample_to_balance(
        X_train,
        y_train.to_numpy(),
        random_state=OVERSAMPLING_RANDOM_STATE,
    )

    model = build_model()
    model.fit(X_train_final, y_train_final)
    val_prob = model.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows_original": orig_train_rows,
            "train_positives_original": orig_train_positives,
            "train_positive_rate_original": float(y_train.mean()),
            "train_rows_model_input": int(len(y_train_final)),
            "train_positives_model_input": int(np.sum(y_train_final)),
            "train_positive_rate_model_input": float(np.mean(y_train_final)),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 13. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev_raw = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy().astype(int)

X_test_raw = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy().astype(int)

preprocessor = build_preprocessor()
X_dev = preprocessor.fit_transform(X_dev_raw, y_dev)
X_test = preprocessor.transform(X_test_raw)

orig_dev_rows = X_dev.shape[0]
orig_dev_positives = int(y_dev.sum())


X_dev_final, y_dev_final = random_oversample_to_balance(
    X_dev,
    y_dev.to_numpy(),
    random_state=OVERSAMPLING_RANDOM_STATE,
)

final_model = build_model()
final_model.fit(X_dev_final, y_dev_final)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 14. SAVE FINAL COEFFICIENTS
# =========================================================
feature_names = clean_feature_names(preprocessor.get_feature_names_out())
coef_df = pd.DataFrame(
    {
        "feature_name": feature_names,
        "coefficient": final_model.coef_.ravel(),
        "odds_ratio": np.exp(final_model.coef_.ravel()),
        "abs_coefficient": np.abs(final_model.coef_.ravel()),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 15. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "random_oversampling",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows_original": np.nan,
            "positives_original": np.nan,
            "positive_rate_original": np.nan,
            "rows_model_input": np.nan,
            "positives_model_input": np.nan,
            "positive_rate_model_input": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "random_oversampling",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows_original": np.nan,
            "positives_original": np.nan,
            "positive_rate_original": np.nan,
            "rows_model_input": np.nan,
            "positives_model_input": np.nan,
            "positive_rate_model_input": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "random_oversampling",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows_original": int(orig_dev_rows),
            "positives_original": int(orig_dev_positives),
            "positive_rate_original": float(y_dev.mean()),
            "rows_model_input": int(len(y_dev_final)),
            "positives_model_input": int(np.sum(y_dev_final)),
            "positive_rate_model_input": float(np.mean(y_dev_final)),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - start_time) / 60.0

print("I1 Linear Winner + Random Oversampling: complete.")
print(f"Total runtime: {total_runtime_minutes:.2f} minutes")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print(f"Saved coefficients to: {COEFFICIENTS_CSV}")
print()
print("Locked base model:")
print("  L4 Elastic-Net Winsor")
print(f"  C = {SELECTED_C}")
print(f"  l1_ratio = {SELECTED_L1_RATIO}")
print("Imbalance treatment: random_oversampling")

I1 Linear Winner + Random Oversampling: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=24053 | val_rows=3888
[CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=27941 | val_rows=3811
[CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=31752 | val_rows=3794
[CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=35546 | val_rows=3805
[CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=39351 | val_rows=3907
[CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=43258 | val_rows=4535
I1 Linear Winner + Random Oversampling: complete.
Total runtime: 0.36 minutes
Saved selected-HP fold metrics to: Model_I1_LinearWinner_RandomOversampling_v1_SelectedHP_DevCV_Fold_Metrics.csv
Saved stage metrics to: Model_I1_LinearWinner_RandomOversampling_v1_Stage_Metrics.csv
Saved predictions to: Model_I1_LinearWinner_RandomOversampling_v1_Predictions.csv
Saved coefficients to: Model_I1_LinearWinner_RandomOversampling_v1_Coefficients.csv

Locked base mode

In [3]:
# Imbalance Extension: I2 Linear Winner + Native Class Weighting
# Based on winning linear family model under final horse-race plan v3
# Base model = Elastic-Net Logistic (winsorized), C=0.1, l1_ratio=0.5
# This script evaluates the fixed winner under a new imbalance treatment only.

import warnings
warnings.filterwarnings(
    "ignore",
    message=r".*'penalty' was deprecated.*",
    category=FutureWarning,
)

import pandas as pd
import numpy as np
from pathlib import Path
import time
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sklearn.linear_model import LogisticRegression

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")

PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

CV_FOLD_METRICS_CSV = BASE_DIR / "Model_I2_LinearWinner_ClassWeight_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_I2_LinearWinner_ClassWeight_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_I2_LinearWinner_ClassWeight_v1_Predictions.csv"
COEFFICIENTS_CSV = BASE_DIR / "Model_I2_LinearWinner_ClassWeight_v1_Coefficients.csv"

# Locked winning linear family hyperparameters
SELECTED_C = 0.1
SELECTED_L1_RATIO = 0.5
MODEL_RANDOM_STATE = 42
OVERSAMPLING_RANDOM_STATE = 42

# =========================================================
# 2. LOAD FROZEN PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED TARGET + SPLITS
# =========================================================
TARGET_COL = "targeted_in_year"

def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"

df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET (SELF-CONTAINED, MATCHES LINEAR v4)
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",          # omitted reference
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. THESIS-STANDARD METRICS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }

# =========================================================
# 6. TRAIN-ONLY WINSORIZER
# =========================================================
class PercentileWinsorizer(BaseEstimator, TransformerMixin):
    """
    Column-wise winsorization using training-sample percentiles only.
    NaNs are ignored when fitting percentiles and preserved until imputation.
    """
    def __init__(self, lower=1.0, upper=99.0):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        n_cols = X.shape[1]
        lower_bounds = []
        upper_bounds = []

        for j in range(n_cols):
            col = X[:, j]
            non_missing = col[~np.isnan(col)]
            if non_missing.size == 0:
                lower_bounds.append(np.nan)
                upper_bounds.append(np.nan)
            else:
                lower_bounds.append(np.percentile(non_missing, self.lower))
                upper_bounds.append(np.percentile(non_missing, self.upper))

        self.lower_bounds_ = np.array(lower_bounds, dtype=float)
        self.upper_bounds_ = np.array(upper_bounds, dtype=float)
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float).copy()
        for j in range(X.shape[1]):
            lb = self.lower_bounds_[j]
            ub = self.upper_bounds_[j]
            if np.isnan(lb) or np.isnan(ub):
                continue
            mask = ~np.isnan(X[:, j])
            X[mask, j] = np.clip(X[mask, j], lb, ub)
        return X

    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# =========================================================
# 7. PREPROCESSOR BUILDER
# =========================================================
def build_preprocessor():
    continuous_transformer = Pipeline(
        steps=[
            ("winsorizer", PercentileWinsorizer(lower=1.0, upper=99.0)),
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    binary_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ]
    )

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

# =========================================================
# 8. MODEL BUILDER
# =========================================================
def build_model():
    return LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        C=SELECTED_C,
        l1_ratio=SELECTED_L1_RATIO,
        max_iter=50000,
        class_weight="balanced", random_state=MODEL_RANDOM_STATE,
    )

# =========================================================
# 9. RANDOM OVERSAMPLING HELPER
# =========================================================
def random_oversample_to_balance(X, y, random_state=42):
    X = np.asarray(X)
    y = np.asarray(y).astype(int)

    class_0_idx = np.where(y == 0)[0]
    class_1_idx = np.where(y == 1)[0]

    if len(class_0_idx) == 0 or len(class_1_idx) == 0:
        return X, y

    majority_idx = class_0_idx if len(class_0_idx) >= len(class_1_idx) else class_1_idx
    minority_idx = class_1_idx if len(class_0_idx) >= len(class_1_idx) else class_0_idx

    n_majority = len(majority_idx)
    n_minority = len(minority_idx)

    if n_majority == n_minority:
        return X, y

    rng = np.random.default_rng(random_state)
    sampled_minority_idx = rng.choice(minority_idx, size=n_majority, replace=True)

    balanced_idx = np.concatenate([majority_idx, sampled_minority_idx])
    rng.shuffle(balanced_idx)

    return X[balanced_idx], y[balanced_idx]

# =========================================================
# 10. FEATURE NAME CLEANUP
# =========================================================
def clean_feature_names(feature_names):
    cleaned = []
    for name in feature_names:
        cleaned_name = name
        for prefix in ["cont__", "bin__", "ff17__"]:
            if cleaned_name.startswith(prefix):
                cleaned_name = cleaned_name[len(prefix):]
        cleaned.append(cleaned_name)
    return cleaned

# =========================================================
# 11. FINAL SIX-FOLD EXPANDING-WINDOW CV
# =========================================================
cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 12. EVALUATE FIXED SELECTED MODEL UNDER NEW IMBALANCE TREATMENT
# =========================================================
start_time = time.time()
selected_cv_rows = []
prediction_frames = []

print("I2 Linear Winner + Native Class Weighting: starting six-fold evaluation...")

for fold_num, fold_def in enumerate(cv_fold_defs, start=1):
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train_raw = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy().astype(int)

    X_val_raw = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy().astype(int)

    print(
        f"[CV] Fold {fold_num}/6 | train={min(train_years)}-{max(train_years)} "
        f"| val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)}"
    )

    preprocessor = build_preprocessor()
    X_train = preprocessor.fit_transform(X_train_raw, y_train)
    X_val = preprocessor.transform(X_val_raw)

    orig_train_rows = X_train.shape[0]
    orig_train_positives = int(y_train.sum())


    X_train_final = X_train
    y_train_final = y_train.to_numpy()

    model = build_model()
    model.fit(X_train_final, y_train_final)
    val_prob = model.predict_proba(X_val)[:, 1]

    fold_metrics = evaluate_predictions(y_val, val_prob)
    fold_metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows_original": orig_train_rows,
            "train_positives_original": orig_train_positives,
            "train_positive_rate_original": float(y_train.mean()),
            "train_rows_model_input": int(len(y_train_final)),
            "train_positives_model_input": int(np.sum(y_train_final)),
            "train_positive_rate_model_input": float(np.mean(y_train_final)),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    selected_cv_rows.append(fold_metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

selected_cv_df = pd.DataFrame(selected_cv_rows)
selected_cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 13. FINAL LOCKED TEST: REFIT ON 2010-2021, TEST ON 2022-2024
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev_raw = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy().astype(int)

X_test_raw = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy().astype(int)

preprocessor = build_preprocessor()
X_dev = preprocessor.fit_transform(X_dev_raw, y_dev)
X_test = preprocessor.transform(X_test_raw)

orig_dev_rows = X_dev.shape[0]
orig_dev_positives = int(y_dev.sum())


X_dev_final = X_dev
y_dev_final = y_dev.to_numpy()

final_model = build_model()
final_model.fit(X_dev_final, y_dev_final)

test_prob = final_model.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 14. SAVE FINAL COEFFICIENTS
# =========================================================
feature_names = clean_feature_names(preprocessor.get_feature_names_out())
coef_df = pd.DataFrame(
    {
        "feature_name": feature_names,
        "coefficient": final_model.coef_.ravel(),
        "odds_ratio": np.exp(final_model.coef_.ravel()),
        "abs_coefficient": np.abs(final_model.coef_.ravel()),
    }
).sort_values("abs_coefficient", ascending=False).reset_index(drop=True)
coef_df.to_csv(COEFFICIENTS_CSV, index=False)

# =========================================================
# 15. STAGE METRICS TABLE
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "class_weight_balanced",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(selected_cv_df["pr_auc"].mean()),
            "roc_auc": float(selected_cv_df["roc_auc"].mean()),
            "brier_score": float(selected_cv_df["brier_score"].mean()),
            "rows_original": np.nan,
            "positives_original": np.nan,
            "positive_rate_original": np.nan,
            "rows_model_input": np.nan,
            "positives_model_input": np.nan,
            "positive_rate_model_input": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "class_weight_balanced",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(selected_cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(selected_cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(selected_cv_df["brier_score"].std(ddof=1)),
            "rows_original": np.nan,
            "positives_original": np.nan,
            "positive_rate_original": np.nan,
            "rows_model_input": np.nan,
            "positives_model_input": np.nan,
            "positive_rate_model_input": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "base_linear_model": "L4_ElasticNet_Winsor",
            "imbalance_treatment": "class_weight_balanced",
            "selected_C": SELECTED_C,
            "selected_l1_ratio": SELECTED_L1_RATIO,
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows_original": int(orig_dev_rows),
            "positives_original": int(orig_dev_positives),
            "positive_rate_original": float(y_dev.mean()),
            "rows_model_input": int(len(y_dev_final)),
            "positives_model_input": int(np.sum(y_dev_final)),
            "positive_rate_model_input": float(np.mean(y_dev_final)),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

total_runtime_minutes = (time.time() - start_time) / 60.0

print("I2 Linear Winner + Native Class Weighting: complete.")
print(f"Total runtime: {total_runtime_minutes:.2f} minutes")
print(f"Saved selected-HP fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print(f"Saved coefficients to: {COEFFICIENTS_CSV}")
print()
print("Locked base model:")
print("  L4 Elastic-Net Winsor")
print(f"  C = {SELECTED_C}")
print(f"  l1_ratio = {SELECTED_L1_RATIO}")
print("Imbalance treatment: class_weight_balanced")

I2 Linear Winner + Native Class Weighting: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=24053 | val_rows=3888


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


[CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=27941 | val_rows=3811
[CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=31752 | val_rows=3794
[CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=35546 | val_rows=3805
[CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=39351 | val_rows=3907
[CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=43258 | val_rows=4535
I2 Linear Winner + Native Class Weighting: complete.
Total runtime: 10.54 minutes
Saved selected-HP fold metrics to: Model_I2_LinearWinner_ClassWeight_v1_SelectedHP_DevCV_Fold_Metrics.csv
Saved stage metrics to: Model_I2_LinearWinner_ClassWeight_v1_Stage_Metrics.csv
Saved predictions to: Model_I2_LinearWinner_ClassWeight_v1_Predictions.csv
Saved coefficients to: Model_I2_LinearWinner_ClassWeight_v1_Coefficients.csv

Locked base model:
  L4 Elastic-Net Winsor
  C = 0.1
  l1_ratio = 0.5
Imbalance treatment: class_weight_balanced


In [4]:
# Model I3 v1: Updated nonlinear winner (XGBoost no-winsor) + random oversampling
# Locked winner specification from revised N3 random-search rerun
# Final six-fold expanding-window evaluation through 2021; 2022-2024 untouched final holdout

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

CV_FOLD_METRICS_CSV = BASE_DIR / "Model_I3_NonlinearWinner_RandomOversampling_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_I3_NonlinearWinner_RandomOversampling_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_I3_NonlinearWinner_RandomOversampling_v1_Predictions.csv"
MODEL_SPEC_CSV = BASE_DIR / "Model_I3_NonlinearWinner_RandomOversampling_v1_Model_Spec.csv"

TARGET_COL = "targeted_in_year"
XGB_RANDOM_STATE = 42
OVERSAMPLING_RANDOM_STATE = 42

# Locked updated N3 winner hyperparameters
SELECTED_N_ESTIMATORS = 800
SELECTED_MAX_DEPTH = 3
SELECTED_LEARNING_RATE = 0.01
SELECTED_MIN_CHILD_WEIGHT = 10
SELECTED_SUBSAMPLE = 0.6
SELECTED_COLSAMPLE_BYTREE = 0.5

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED SPLITS
# =========================================================
def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"


df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. HELPERS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


def random_oversample_training_rows(train_df: pd.DataFrame, target_col: str, random_state: int) -> pd.DataFrame:
    positives = train_df[train_df[target_col] == 1]
    negatives = train_df[train_df[target_col] == 0]

    n_pos = len(positives)
    n_neg = len(negatives)

    if n_pos == 0 or n_pos >= n_neg:
        return train_df.copy()

    rng = np.random.default_rng(random_state)
    sampled_idx = rng.choice(positives.index.to_numpy(), size=(n_neg - n_pos), replace=True)
    sampled_positives = train_df.loc[sampled_idx].copy()

    oversampled = pd.concat([train_df, sampled_positives], axis=0, ignore_index=True)
    oversampled = oversampled.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
    return oversampled


def build_pipeline() -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )
    binary_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=SELECTED_N_ESTIMATORS,
        max_depth=SELECTED_MAX_DEPTH,
        learning_rate=SELECTED_LEARNING_RATE,
        min_child_weight=SELECTED_MIN_CHILD_WEIGHT,
        subsample=SELECTED_SUBSAMPLE,
        colsample_bytree=SELECTED_COLSAMPLE_BYTREE,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )


cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 6. SIX-FOLD EVALUATION
# =========================================================
start_time = time.time()
print("I3 Nonlinear Winner + Random Oversampling: starting six-fold evaluation...")

cv_rows = []
prediction_frames = []

for i, fold_def in enumerate(cv_fold_defs, start=1):
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df_raw = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    train_df = random_oversample_training_rows(
        train_df_raw,
        target_col=TARGET_COL,
        random_state=OVERSAMPLING_RANDOM_STATE + i,
    )

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()

    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    raw_pos = int(train_df_raw[TARGET_COL].sum())
    raw_neg = int((train_df_raw[TARGET_COL] == 0).sum())
    os_pos = int(y_train.sum())
    os_neg = int((y_train == 0).sum())

    print(
        f"[CV] Fold {i}/{len(cv_fold_defs)} | train={min(train_years)}-{max(train_years)} | "
        f"val={min(val_years)} | raw_train_rows={len(train_df_raw)} | os_train_rows={len(train_df)} | "
        f"raw_pos={raw_pos} | raw_neg={raw_neg} | os_pos={os_pos} | os_neg={os_neg}"
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    metrics = evaluate_predictions(y_val, val_prob)
    metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "raw_train_rows": len(train_df_raw),
            "raw_train_positives": raw_pos,
            "raw_train_positive_rate": float(train_df_raw[TARGET_COL].mean()),
            "oversampled_train_rows": len(train_df),
            "oversampled_train_positives": os_pos,
            "oversampled_train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_rows.append(metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
            }
        )
    )

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 7. FINAL TEST: REFIT ON FULL DEV WITH OVERSAMPLING
# =========================================================
dev_df_raw = df[df["year"].between(2010, 2021)].copy()
dev_df = random_oversample_training_rows(
    dev_df_raw,
    target_col=TARGET_COL,
    random_state=OVERSAMPLING_RANDOM_STATE + 999,
)
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

print(
    f"[FINAL] train=2010-2021 | raw_train_rows={len(dev_df_raw)} | os_train_rows={len(dev_df)} | "
    f"raw_pos={int(dev_df_raw[TARGET_COL].sum())} | raw_neg={int((dev_df_raw[TARGET_COL] == 0).sum())} | "
    f"os_pos={int(y_dev.sum())} | os_neg={int((y_dev == 0).sum())}"
)

final_pipeline = build_pipeline()
final_pipeline.fit(X_dev, y_dev)
test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 8. STAGE METRICS + MODEL SPEC
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "random_oversampling",
            "pr_auc": float(cv_df["pr_auc"].mean()),
            "roc_auc": float(cv_df["roc_auc"].mean()),
            "brier_score": float(cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "random_oversampling",
            "pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "random_oversampling",
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

spec_df = pd.DataFrame(
    [
        {"parameter": "base_model", "value": "updated_N3_xgb_no_winsor"},
        {"parameter": "imbalance_treatment", "value": "random_oversampling"},
        {"parameter": "n_estimators", "value": SELECTED_N_ESTIMATORS},
        {"parameter": "max_depth", "value": SELECTED_MAX_DEPTH},
        {"parameter": "learning_rate", "value": SELECTED_LEARNING_RATE},
        {"parameter": "min_child_weight", "value": SELECTED_MIN_CHILD_WEIGHT},
        {"parameter": "subsample", "value": SELECTED_SUBSAMPLE},
        {"parameter": "colsample_bytree", "value": SELECTED_COLSAMPLE_BYTREE},
        {"parameter": "xgb_random_state", "value": XGB_RANDOM_STATE},
        {"parameter": "oversampling_random_state", "value": OVERSAMPLING_RANDOM_STATE},
    ]
)
spec_df.to_csv(MODEL_SPEC_CSV, index=False)

elapsed_minutes = (time.time() - start_time) / 60.0
print("I3 Nonlinear Winner + Random Oversampling: complete.")
print(f"Total runtime: {elapsed_minutes:.2f} minutes")
print(f"Saved fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print(f"Saved model spec to: {MODEL_SPEC_CSV}")

I3 Nonlinear Winner + Random Oversampling: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | raw_train_rows=24053 | os_train_rows=46294 | raw_pos=906 | raw_neg=23147 | os_pos=23147 | os_neg=23147
[CV] Fold 2/6 | train=2010-2016 | val=2017 | raw_train_rows=27941 | os_train_rows=53682 | raw_pos=1100 | raw_neg=26841 | os_pos=26841 | os_neg=26841
[CV] Fold 3/6 | train=2010-2017 | val=2018 | raw_train_rows=31752 | os_train_rows=60888 | raw_pos=1308 | raw_neg=30444 | os_pos=30444 | os_neg=30444
[CV] Fold 4/6 | train=2010-2018 | val=2019 | raw_train_rows=35546 | os_train_rows=68052 | raw_pos=1520 | raw_neg=34026 | os_pos=34026 | os_neg=34026
[CV] Fold 5/6 | train=2010-2019 | val=2020 | raw_train_rows=39351 | os_train_rows=75326 | raw_pos=1688 | raw_neg=37663 | os_pos=37663 | os_neg=37663
[CV] Fold 6/6 | train=2010-2020 | val=2021 | raw_train_rows=43258 | os_train_rows=82722 | raw_pos=1897 | raw_neg=41361 | os_pos=41361 | os_neg=41361
[FINAL] train=2010-2021 | raw_tr

In [5]:
# Model I4 v1: Updated nonlinear winner (XGBoost no-winsor) + native class weighting
# Locked winner specification from revised N3 random-search rerun
# Final six-fold expanding-window evaluation through 2021; 2022-2024 untouched final holdout

import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

# =========================================================
# 1. USER SETTINGS
# =========================================================
BASE_DIR = Path(".")
PANEL_PATH = BASE_DIR / "Master_Modeling_Panel_v2.csv"

CV_FOLD_METRICS_CSV = BASE_DIR / "Model_I4_NonlinearWinner_ClassWeight_v1_SelectedHP_DevCV_Fold_Metrics.csv"
STAGE_METRICS_CSV = BASE_DIR / "Model_I4_NonlinearWinner_ClassWeight_v1_Stage_Metrics.csv"
PREDICTIONS_CSV = BASE_DIR / "Model_I4_NonlinearWinner_ClassWeight_v1_Predictions.csv"
MODEL_SPEC_CSV = BASE_DIR / "Model_I4_NonlinearWinner_ClassWeight_v1_Model_Spec.csv"

TARGET_COL = "targeted_in_year"
XGB_RANDOM_STATE = 42

# Locked updated N3 winner hyperparameters
SELECTED_N_ESTIMATORS = 800
SELECTED_MAX_DEPTH = 3
SELECTED_LEARNING_RATE = 0.01
SELECTED_MIN_CHILD_WEIGHT = 10
SELECTED_SUBSAMPLE = 0.6
SELECTED_COLSAMPLE_BYTREE = 0.5

# =========================================================
# 2. LOAD PANEL
# =========================================================
df = pd.read_csv(PANEL_PATH, low_memory=False)

# =========================================================
# 3. LOCKED SPLITS
# =========================================================
def assign_split(year: int) -> str:
    if 2010 <= year <= 2021:
        return "development"
    if 2022 <= year <= 2024:
        return "final_test"
    return "exclude"


df["split"] = df["year"].apply(assign_split)
df = df[df["split"] != "exclude"].copy()

# =========================================================
# 4. LOCKED RAW PREDICTOR SET
# =========================================================
continuous_features = [
    "roe",
    "assets_to_equity",
    "current_ratio",
    "ebitda",
    "ebitda_margin",
    "sales_to_assets",
    "sales_growth",
    "interest_coverage",
    "net_debt_to_ebitda",
    "fcf_to_capex",
    "market_cap",
    "ret_3m",
    "ret_6m",
    "ret_1y",
    "ret_2y",
    "ret_3y",
    "ret_5y",
    "volatility_30d",
    "volatility_90d",
    "volatility_180d",
    "turnover_30d",
    "dividend_payout_ratio",
    "buyback_yield",
    "price_to_book",
    "ev_to_sales",
    "ev_to_ebitda",
    "tobins_q",
    "fcf_yield",
    "prior_campaign_count_3y",
    "prior_settlement_count_3y",
    "prior_management_favorable_count_3y",
    "prior_dissident_favorable_count_3y",
    "prior_campaign_count_5y",
    "prior_settlement_count_5y",
    "prior_management_favorable_count_5y",
    "prior_dissident_favorable_count_5y",
    "ff49_other_permno_years_in_category",
    "ret_3m_rel_peer",
    "ret_6m_rel_peer",
    "ret_1y_rel_peer",
    "ret_2y_rel_peer",
    "ret_3y_rel_peer",
    "ret_5y_rel_peer",
    "log_ev_to_sales_rel_peer",
    "log_price_to_book_rel_peer",
    "log_tobins_q_rel_peer",
    "log_ev_to_ebitda_rel_peer",
    "ebitda_margin_rel_peer",
    "sales_growth_rel_peer",
    "roe_rel_peer",
    "board_size",
    "board_female_ratio",
    "ceo_tenure",
    "top10_inst_conc_within_13f",
    "inst_ownership_pct_13f",
]

binary_features = [
    "prior_activism_3y",
    "prior_activism_5y",
    "history_3y_observed",
    "history_5y_observed",
    "ff49_unclassified",
    "classified_board",
    "poison_pill",
    "dual_class",
    "iss_match_found",
    "rm_stale_gt_730",
    "board_match_found",
    "board_stale_gt_365",
    "ceo_is_female",
    "ceo_chair_duality",
    "ceo_match_found",
    "ceo_stale_gt_365",
    "inst_match_found",
]

categorical_feature = "ff17_for_model"
df[categorical_feature] = df["ff17_short"].fillna("Unclassified").astype(str)

ff17_categories = [
    "Cars",
    "Chems",
    "Clths",
    "Cnstr",
    "Cnsum",
    "Durbl",
    "FabPr",
    "Finan",
    "Food",
    "Machn",
    "Mines",
    "Oil",
    "Other",
    "Rtail",
    "Steel",
    "Trans",
    "Utils",
    "Unclassified",
]

required_cols = continuous_features + binary_features + [categorical_feature, TARGET_COL, "year", "permno"]
missing_required = [c for c in required_cols if c not in df.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

all_predictors = continuous_features + binary_features + [categorical_feature]

# =========================================================
# 5. HELPERS
# =========================================================
def evaluate_predictions(y_true: pd.Series, y_prob: np.ndarray) -> dict:
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob, dtype=float)
    return {
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "brier_score": brier_score_loss(y_true, y_prob),
    }


def compute_scale_pos_weight(y: pd.Series) -> float:
    y = np.asarray(y).astype(int)
    positives = float((y == 1).sum())
    negatives = float((y == 0).sum())
    if positives <= 0:
        return 1.0
    return negatives / positives


def build_pipeline(scale_pos_weight: float) -> Pipeline:
    continuous_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="median"))]
    )
    binary_transformer = Pipeline(
        steps=[("imputer", SimpleImputer(strategy="constant", fill_value=0))]
    )
    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unclassified")),
            (
                "onehot",
                OneHotEncoder(
                    categories=[ff17_categories],
                    drop=["Other"],
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("cont", continuous_transformer, continuous_features),
            ("bin", binary_transformer, binary_features),
            ("ff17", categorical_transformer, [categorical_feature]),
        ],
        remainder="drop",
    )

    model = XGBClassifier(
        objective="binary:logistic",
        eval_metric="logloss",
        n_estimators=SELECTED_N_ESTIMATORS,
        max_depth=SELECTED_MAX_DEPTH,
        learning_rate=SELECTED_LEARNING_RATE,
        min_child_weight=SELECTED_MIN_CHILD_WEIGHT,
        subsample=SELECTED_SUBSAMPLE,
        colsample_bytree=SELECTED_COLSAMPLE_BYTREE,
        scale_pos_weight=scale_pos_weight,
        random_state=XGB_RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

    return Pipeline(
        steps=[("preprocessor", preprocessor), ("model", model)]
    )


cv_fold_defs = [
    {"fold": "fold_1", "train_years": list(range(2010, 2016)), "val_years": [2016]},
    {"fold": "fold_2", "train_years": list(range(2010, 2017)), "val_years": [2017]},
    {"fold": "fold_3", "train_years": list(range(2010, 2018)), "val_years": [2018]},
    {"fold": "fold_4", "train_years": list(range(2010, 2019)), "val_years": [2019]},
    {"fold": "fold_5", "train_years": list(range(2010, 2020)), "val_years": [2020]},
    {"fold": "fold_6", "train_years": list(range(2010, 2021)), "val_years": [2021]},
]

# =========================================================
# 6. SIX-FOLD EVALUATION
# =========================================================
start_time = time.time()
print("I4 Nonlinear Winner + Native Class Weighting: starting six-fold evaluation...")

cv_rows = []
prediction_frames = []

for i, fold_def in enumerate(cv_fold_defs, start=1):
    fold_name = fold_def["fold"]
    train_years = fold_def["train_years"]
    val_years = fold_def["val_years"]

    train_df = df[df["year"].isin(train_years)].copy()
    val_df = df[df["year"].isin(val_years)].copy()

    X_train = train_df[all_predictors].copy()
    y_train = train_df[TARGET_COL].copy()
    X_val = val_df[all_predictors].copy()
    y_val = val_df[TARGET_COL].copy()

    scale_pos_weight = compute_scale_pos_weight(y_train)

    print(
        f"[CV] Fold {i}/{len(cv_fold_defs)} | train={min(train_years)}-{max(train_years)} | "
        f"val={min(val_years)} | train_rows={len(train_df)} | val_rows={len(val_df)} | "
        f"train_pos={int(y_train.sum())} | train_neg={int((y_train == 0).sum())} | "
        f"scale_pos_weight={scale_pos_weight:.4f}"
    )

    pipeline = build_pipeline(scale_pos_weight=scale_pos_weight)
    pipeline.fit(X_train, y_train)
    val_prob = pipeline.predict_proba(X_val)[:, 1]

    metrics = evaluate_predictions(y_val, val_prob)
    metrics.update(
        {
            "fold": fold_name,
            "train_year_start": min(train_years),
            "train_year_end": max(train_years),
            "val_year_start": min(val_years),
            "val_year_end": max(val_years),
            "train_rows": len(train_df),
            "train_positives": int(y_train.sum()),
            "train_positive_rate": float(y_train.mean()),
            "val_rows": len(val_df),
            "val_positives": int(y_val.sum()),
            "val_positive_rate": float(y_val.mean()),
            "scale_pos_weight": float(scale_pos_weight),
            "mean_predicted_probability": float(np.mean(val_prob)),
        }
    )
    cv_rows.append(metrics)

    prediction_frames.append(
        pd.DataFrame(
            {
                "permno": val_df["permno"].to_numpy(),
                "year": val_df["year"].to_numpy(),
                "evaluation_stage": "development_cv",
                "fold": fold_name,
                "actual": y_val.to_numpy(),
                "predicted_probability": val_prob,
                "scale_pos_weight": scale_pos_weight,
            }
        )
    )

cv_df = pd.DataFrame(cv_rows)
cv_df.to_csv(CV_FOLD_METRICS_CSV, index=False)

# =========================================================
# 7. FINAL TEST: REFIT ON FULL DEV WITH CLASS WEIGHTING
# =========================================================
dev_df = df[df["year"].between(2010, 2021)].copy()
test_df = df[df["year"].between(2022, 2024)].copy()

X_dev = dev_df[all_predictors].copy()
y_dev = dev_df[TARGET_COL].copy()
X_test = test_df[all_predictors].copy()
y_test = test_df[TARGET_COL].copy()

final_scale_pos_weight = compute_scale_pos_weight(y_dev)

print(
    f"[FINAL] train=2010-2021 | train_rows={len(dev_df)} | test_rows={len(test_df)} | "
    f"train_pos={int(y_dev.sum())} | train_neg={int((y_dev == 0).sum())} | "
    f"scale_pos_weight={final_scale_pos_weight:.4f}"
)

final_pipeline = build_pipeline(scale_pos_weight=final_scale_pos_weight)
final_pipeline.fit(X_dev, y_dev)
test_prob = final_pipeline.predict_proba(X_test)[:, 1]
test_metrics = evaluate_predictions(y_test, test_prob)

prediction_frames.append(
    pd.DataFrame(
        {
            "permno": test_df["permno"].to_numpy(),
            "year": test_df["year"].to_numpy(),
            "evaluation_stage": "final_test",
            "fold": "final_test",
            "actual": y_test.to_numpy(),
            "predicted_probability": test_prob,
            "scale_pos_weight": final_scale_pos_weight,
        }
    )
)

predictions_df = pd.concat(prediction_frames, ignore_index=True)
predictions_df.to_csv(PREDICTIONS_CSV, index=False)

# =========================================================
# 8. STAGE METRICS + MODEL SPEC
# =========================================================
stage_metrics_df = pd.DataFrame(
    [
        {
            "stage": "development_cv_mean",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "native_class_weighting_scale_pos_weight",
            "pr_auc": float(cv_df["pr_auc"].mean()),
            "roc_auc": float(cv_df["roc_auc"].mean()),
            "brier_score": float(cv_df["brier_score"].mean()),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "development_cv_std",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "native_class_weighting_scale_pos_weight",
            "pr_auc": float(cv_df["pr_auc"].std(ddof=1)),
            "roc_auc": float(cv_df["roc_auc"].std(ddof=1)),
            "brier_score": float(cv_df["brier_score"].std(ddof=1)),
            "rows": np.nan,
            "positives": np.nan,
            "positive_rate": np.nan,
            "mean_predicted_probability": np.nan,
        },
        {
            "stage": "final_test",
            "model_base": "updated_N3_xgb_no_winsor",
            "imbalance_treatment": "native_class_weighting_scale_pos_weight",
            "pr_auc": float(test_metrics["pr_auc"]),
            "roc_auc": float(test_metrics["roc_auc"]),
            "brier_score": float(test_metrics["brier_score"]),
            "rows": int(len(test_df)),
            "positives": int(y_test.sum()),
            "positive_rate": float(y_test.mean()),
            "mean_predicted_probability": float(np.mean(test_prob)),
        },
    ]
)
stage_metrics_df.to_csv(STAGE_METRICS_CSV, index=False)

spec_df = pd.DataFrame(
    [
        {"parameter": "base_model", "value": "updated_N3_xgb_no_winsor"},
        {"parameter": "imbalance_treatment", "value": "native_class_weighting_scale_pos_weight"},
        {"parameter": "n_estimators", "value": SELECTED_N_ESTIMATORS},
        {"parameter": "max_depth", "value": SELECTED_MAX_DEPTH},
        {"parameter": "learning_rate", "value": SELECTED_LEARNING_RATE},
        {"parameter": "min_child_weight", "value": SELECTED_MIN_CHILD_WEIGHT},
        {"parameter": "subsample", "value": SELECTED_SUBSAMPLE},
        {"parameter": "colsample_bytree", "value": SELECTED_COLSAMPLE_BYTREE},
        {"parameter": "xgb_random_state", "value": XGB_RANDOM_STATE},
        {"parameter": "final_scale_pos_weight", "value": final_scale_pos_weight},
    ]
)
spec_df.to_csv(MODEL_SPEC_CSV, index=False)

elapsed_minutes = (time.time() - start_time) / 60.0
print("I4 Nonlinear Winner + Native Class Weighting: complete.")
print(f"Total runtime: {elapsed_minutes:.2f} minutes")
print(f"Saved fold metrics to: {CV_FOLD_METRICS_CSV}")
print(f"Saved stage metrics to: {STAGE_METRICS_CSV}")
print(f"Saved predictions to: {PREDICTIONS_CSV}")
print(f"Saved model spec to: {MODEL_SPEC_CSV}")

I4 Nonlinear Winner + Native Class Weighting: starting six-fold evaluation...
[CV] Fold 1/6 | train=2010-2015 | val=2016 | train_rows=24053 | val_rows=3888 | train_pos=906 | train_neg=23147 | scale_pos_weight=25.5486
[CV] Fold 2/6 | train=2010-2016 | val=2017 | train_rows=27941 | val_rows=3811 | train_pos=1100 | train_neg=26841 | scale_pos_weight=24.4009
[CV] Fold 3/6 | train=2010-2017 | val=2018 | train_rows=31752 | val_rows=3794 | train_pos=1308 | train_neg=30444 | scale_pos_weight=23.2752
[CV] Fold 4/6 | train=2010-2018 | val=2019 | train_rows=35546 | val_rows=3805 | train_pos=1520 | train_neg=34026 | scale_pos_weight=22.3855
[CV] Fold 5/6 | train=2010-2019 | val=2020 | train_rows=39351 | val_rows=3907 | train_pos=1688 | train_neg=37663 | scale_pos_weight=22.3122
[CV] Fold 6/6 | train=2010-2020 | val=2021 | train_rows=43258 | val_rows=4535 | train_pos=1897 | train_neg=41361 | scale_pos_weight=21.8034
[FINAL] train=2010-2021 | train_rows=47793 | test_rows=13147 | train_pos=2126 | tra